# Hotel Recommendation — Exploratory Data Analysis

Explore the **hotels** dataset used for content-based hotel recommendations.

Recommendation features: `place` + `name` (TF-IDF cosine similarity)

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate voyage-analytics project root")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.data.ingestion import DataIngestion

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load data

In [ ]:
hotels_df = DataIngestion().load_hotels_data()
hotels_df.head()

**Insight:** Each row is one hotel booking linked to a trip (`travelCode`) and user (`userCode`). The recommender uses only `name` + `place` at inference — IDs are for exploration here.

## 2. Dataset overview

In [ ]:
print(f"Rows: {len(hotels_df):,}  |  Columns: {hotels_df.shape[1]}")
print(f"Unique hotels: {hotels_df['name'].nunique()}")
print(f"Unique places: {hotels_df['place'].nunique()}")
print(f"Date range: {hotels_df['date'].min()} → {hotels_df['date'].max()}")

hotels_df.info()
hotels_df.describe().T

**Insight:** ~40k bookings but only **9 hotel names** and **9 destinations** — a small catalog repeated many times. Recommendations will be driven by location/name text similarity, not catalog size.

## 3. Missing values

In [ ]:
missing = hotels_df.isna().sum().sort_values(ascending=False)
pd.DataFrame({"missing": missing, "pct": (missing / len(hotels_df) * 100).round(3)})

**Insight:** No missing values — pricing, stay length, and location fields are complete.

## 4. Hotel catalog

In [ ]:
catalog = (
    hotels_df.groupby(["name", "place"], as_index=False)
    .agg(bookings=("travelCode", "count"), avg_nightly_price=("price", "mean"))
    .sort_values("bookings", ascending=False)
)
catalog["avg_nightly_price"] = catalog["avg_nightly_price"].round(2)
catalog

**Insight:** Each hotel name maps to one place. Booking volume varies — some hotels (e.g. `Hotel K`) appear far more often, which reflects trip patterns, not recommendation logic.

In [ ]:
# Booking volume by hotel name (left) and destination city (right)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

hotels_df["name"].value_counts().plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Bookings by hotel name")
axes[0].set_xlabel("Hotel")
axes[0].set_ylabel("Booking count")
axes[0].tick_params(axis="x", rotation=45)

hotels_df["place"].value_counts().plot(kind="bar", ax=axes[1], color="#F58518")
axes[1].set_title("Bookings by destination")
axes[1].set_xlabel("Place")
axes[1].set_ylabel("Booking count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**Insight:** Left chart — which hotels are booked most. Right chart — which cities drive demand. Imbalance here means some destinations dominate the training signal.

## 5. Pricing & stay length

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(hotels_df["price"], bins=30, color="#4C78A8", edgecolor="white")
axes[0].set_title("Nightly price distribution")
axes[0].set_xlabel("Price per night")

axes[1].hist(hotels_df["days"], bins=range(1, hotels_df["days"].max() + 2), color="#F58518", edgecolor="white")
axes[1].set_title("Length of stay (days)")
axes[1].set_xlabel("Days")

axes[2].hist(hotels_df["total"], bins=30, color="#54A24B", edgecolor="white")
axes[2].set_title("Total booking cost")
axes[2].set_xlabel("Total")

plt.tight_layout()
plt.show()

**Insight:** Nightly `price` and `total` are right-skewed; `days` (stay length) clusters at short trips (1–4 nights). Total cost = price × days in most rows.

In [ ]:
price_by_place = hotels_df.groupby("place")[["price", "days", "total"]].mean().round(2)
price_by_place.sort_values("price", ascending=False)

**Insight:** Average nightly rate differs by city — expensive destinations will surface different recommendation neighborhoods in TF-IDF space.

In [ ]:
hotels_df.boxplot(column="price", by="place", figsize=(10, 4), rot=45)
plt.title("Nightly price by destination")
plt.suptitle("")
plt.xlabel("Place")
plt.ylabel("Price per night")
plt.tight_layout()
plt.show()

**Insight:** Boxplots show price spread within each city. Wide boxes mean the same destination can have very different hotel price tiers.

## 6. User & travel activity

In [ ]:
user_activity = hotels_df.groupby("userCode").size()
travel_activity = hotels_df.groupby("travelCode").size()

print(f"Unique users: {hotels_df['userCode'].nunique():,}")
print(f"Unique trips: {hotels_df['travelCode'].nunique():,}")
print(f"Avg hotel bookings per user: {user_activity.mean():.2f}")
print(f"Avg hotel bookings per trip: {travel_activity.mean():.2f}")

**Insight:** Most users book multiple hotels across trips; activity is repeat-heavy. This supports collaborative filtering in future work, though the current model is content-based only.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

user_activity.plot(kind="hist", bins=30, ax=axes[0], color="#4C78A8", edgecolor="white")
axes[0].set_title("Bookings per user")
axes[0].set_xlabel("Number of bookings")

travel_activity.plot(kind="hist", bins=30, ax=axes[1], color="#F58518", edgecolor="white")
axes[1].set_title("Bookings per trip")
axes[1].set_xlabel("Number of bookings")

plt.tight_layout()
plt.show()

**Insight:** Histograms show how booking frequency is distributed. A long tail of heavy users exists — they are not used by the current recommender.

## 7. Recommendation feature space

The production recommender deduplicates hotels by `name` and builds TF-IDF vectors from `place + name`.

In [ ]:
unique_hotels = hotels_df.drop_duplicates(subset="name").copy()
unique_hotels["combined_features"] = (
    unique_hotels["place"].astype(str) + " " + unique_hotels["name"].astype(str)
)

print(f"Unique hotel entities for similarity: {len(unique_hotels)}")
unique_hotels[["name", "place", "combined_features"]].sort_values("name")

**Insight:** Only **9 unique hotel entities** after deduplication by `name`. The TF-IDF feature space is tiny — similar hotels will share destination tokens.

In [ ]:
# Preview the same TF-IDF + cosine similarity logic used in production
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(unique_hotels["combined_features"])
similarity = cosine_similarity(tfidf_matrix)

example_hotel = unique_hotels.iloc[0]["name"]
idx = unique_hotels.index[0]
scores = list(enumerate(similarity[idx]))
scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:4]

print(f"Top similar hotels to '{example_hotel}':")
for i, score in scores:
    row = unique_hotels.loc[i]
    print(f"  - {row['name']} ({row['place']})  similarity={score:.3f}")

**Insight:** Cosine similarity preview shows which hotels the model would recommend. Hotels in the same or nearby cities score highest because `place` dominates the text features.

## 8. Modeling notes

- This is a **content-based** system (not collaborative filtering); user/trip IDs are not used at inference.
- Only 9 unique hotel names exist, so recommendations are driven mainly by destination text overlap.
- For richer recommendations, consider adding amenities, ratings, or user interaction matrices.

In [ ]:
summary = {
    "records": len(hotels_df),
    "unique_hotels": hotels_df["name"].nunique(),
    "unique_places": hotels_df["place"].nunique(),
    "median_nightly_price": round(hotels_df["price"].median(), 2),
    "median_stay_days": hotels_df["days"].median(),
    "unique_users": hotels_df["userCode"].nunique(),
    "unique_trips": hotels_df["travelCode"].nunique(),
}
pd.Series(summary, name="hotel_recommendation_eda")

**Insight — key takeaways:** Content-based TF-IDF on 9 hotels limits diversity. `place` drives similarity more than `name`. Consider ratings, amenities, or collaborative filtering to enrich recommendations.